# Top n de documentos recuperados por modelo (consulta de prueba)

Dado un **número de consulta** de la partición de prueba de MessIRve, este cuaderno devuelve los **n documentos mejor posicionados** (10 por omisión) de cada **modelo de recuperación inicial** de los experimentos, en el orden en que ese modelo los rankea.

Los rankings se leen de los *runs* que ya calculó el pipeline de recuperación (`~/.cache/messirve_embeddings/<modelo>/<configuración>/retrieval_run.lz4`): el cuaderno **no vuelve a recuperar** nada, no codifica embeddings ni usa GPU. Se ejecuta con el entorno `proyecto` (conda), que es el que tiene `ranx`, `datasets` y `pandas`.

**Modelos incluidos** (seis, ver §2): BM25, SPLADE-v3, multilingual-e5-large-instruct, BGE-M3, Qwen3-Embedding-0.6B y jina-embeddings-v5-text-small-retrieval. `microsoft/harrier-oss-v1-0.6b` **queda excluido** a propósito.

**Uso.** Fija `NUMERO_CONSULTA` en la §1 (es el `id` numérico de la consulta en el split de prueba; el ejemplo es `8101866`) y ejecuta el cuaderno completo. Para inspeccionar otra consulta, cambia el número en la §1 y vuelve a ejecutar desde la §4; para *encontrar* un número están `listar_consultas()` y `buscar_consultas()` en la §3.

## 1. Configuración

- `NUMERO_CONSULTA`: número (`id`) de la consulta de prueba que se quiere inspeccionar.
- `N`: cuántos documentos devuelve cada modelo.
- `INCLUIR_TEXTO`: si es `True`, la §5 lee el corpus de párrafos de la Wikipedia en español para añadir el título y un extracto de cada documento; si es `False`, las tablas de la §6 salen solo con `docid` y puntaje.

In [ ]:
import gc
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------
NUMERO_CONSULTA = 8101866   # número (id) de la consulta de prueba a inspeccionar; ver §3
N = 10                      # documentos que devuelve cada modelo
INCLUIR_TEXTO = True        # True: título y extracto de cada documento (lee el corpus, §5)


# ---------------------------------------------------------------------------
# Entorno
# ---------------------------------------------------------------------------
def _raiz_repositorio() -> Path:
    """Carpeta de ir-spanish/ (la que contiene utils/), para importar el código del pipeline.

    El cuaderno vive en analysis/, pero JupyterLab lo abre con ese directorio como cwd y
    nbconvert lo abre donde se lance, así que la raíz se busca hacia arriba en vez de suponerla.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "utils" / "cache.py").exists():
            return base
    raise RuntimeError(
        "No se encontró la raíz de ir-spanish (la carpeta que contiene utils/). "
        "Abre el cuaderno desde el repositorio o ejecútalo desde su raíz."
    )


RAIZ = _raiz_repositorio()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from ranx.io import load_lz4          # lee un run tal como lo escribió ranx.Run.save
from utils import cache, data         # rutas de la caché y constantes del conjunto de datos

CACHE_DIR = Path.home() / ".cache" / "messirve_embeddings"

print(f"Repositorio:   {RAIZ}")
print(f"Caché de runs: {CACHE_DIR}")
print(f"Consulta:      {NUMERO_CONSULTA}   |   n = {N}   |   texto de los documentos: {INCLUIR_TEXTO}")

## 2. Modelos de recuperación inicial

Cada modelo tiene su *run* cacheado. La ruta se deriva con `utils.cache` —la misma que usa el pipeline— a partir de la configuración con la que se generó el run (`max_query_length` y `max_doc_length` de la tabla de resultados). BM25 no tiene longitudes de secuencia, así que `baselines/bm25.py` guarda su run en el directorio que solo depende del conjunto de datos; el mismo archivo está también en la ruta con longitudes (idénticos byte a byte), y se acepta cualquiera de las dos.

`microsoft/harrier-oss-v1-0.6b` **no se incluye** (petición del usuario, 2026-09-25). Queda en el registro como línea comentada, para que la exclusión sea explícita y no un olvido.

In [ ]:
@dataclass(frozen=True)
class ModeloInicial:
    """Un modelo de recuperación inicial y la configuración con la que se cacheó su run."""
    nombre: str         # nombre en HuggingFace o identificador del pipeline
    alias: str          # nombre corto para las tablas
    max_q: int | None   # max_query_length del run (None: el modelo no trunca por longitudes)
    max_d: int | None   # max_doc_length del run


MODELOS = [
    # Léxico
    ModeloInicial("bm25_pyserini", "bm25", None, None),
    # Disperso (learned sparse)
    ModeloInicial("naver/splade-v3", "splade-v3", 512, 512),
    # Densos (dual-encoders)
    ModeloInicial("intfloat/multilingual-e5-large-instruct", "e5-large", 512, 512),
    ModeloInicial("BAAI/bge-m3", "bge-m3", 8192, 8192),
    ModeloInicial("Qwen/Qwen3-Embedding-0.6B", "qwen3-0.6b", 32768, 32768),
    ModeloInicial("jinaai/jina-embeddings-v5-text-small-retrieval", "jina-v5-small", 32768, 32768),
    # Excluido a propósito del análisis (2026-09-25):
    # ModeloInicial("microsoft/harrier-oss-v1-0.6b", "harrier", 32768, 32768),
]


def rutas_run(modelo: ModeloInicial) -> list[Path]:
    """Rutas donde puede estar el run de un modelo, en orden de preferencia.

    BM25 no tiene longitudes de secuencia: `baselines/bm25.py` guarda su run en el directorio
    que solo depende del conjunto de datos, y el mismo archivo se copió a la ruta con longitudes,
    que es la que usan los scripts de fusión. Se acepta cualquiera de las dos.
    """
    if modelo.max_q is None:
        solo_conjunto = CACHE_DIR / cache.model_slug(modelo.nombre) / (
            f"{data.COUNTRY}_v{data.DATASET_VERSION}_{cache._filter_suffix(data.MAX_WORD_COUNT)}"
        )
        con_longitudes = cache.cache_base(
            CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
            512, 512, data.MAX_WORD_COUNT,
        )
        return [cache.run_cache_path(solo_conjunto), cache.run_cache_path(con_longitudes)]

    base = cache.cache_base(
        CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
        modelo.max_q, modelo.max_d, data.MAX_WORD_COUNT,
    )
    return [cache.run_cache_path(base)]


def ruta_run(modelo: ModeloInicial) -> Path:
    """Primera ruta existente del run; falla con un mensaje claro si no hay ninguna."""
    candidatas = rutas_run(modelo)
    for ruta in candidatas:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No hay run cacheado para {modelo.nombre}. Rutas probadas:\n  "
        + "\n  ".join(str(ruta) for ruta in candidatas)
    )


# Comprobación: de qué archivo se va a leer cada modelo
filas = []
for modelo in MODELOS:
    run = ruta_run(modelo)
    filas.append({
        "alias": modelo.alias,
        "modelo": modelo.nombre,
        "max_query_length, max_doc_length": "—" if modelo.max_q is None else f"{modelo.max_q}, {modelo.max_d}",
        "run": str(run.relative_to(CACHE_DIR)),
        "MB": round(run.stat().st_size / 1024**2, 1),
    })

display(pd.DataFrame(filas))

## 3. La consulta de prueba y su verdad de referencia

El **número de consulta** es el campo `id` del split de prueba (un entero, p. ej. `8101866`); las claves de los runs y de la verdad de referencia son ese número en forma de cadena. La partición de prueba tiene 170,055 consultas únicas.

La verdad de referencia se lee de `pruned_qrels.json`, el mismo archivo con el que el pipeline evalúa: para cada consulta, los `docid` relevantes. Un documento es un párrafo de un artículo de Wikipedia, identificado por un `docid` de la forma `"articulo#parrafo"` (p. ej. `"328242#0"`).

Para elegir un número: `listar_consultas(inicio, cuantas)` recorre la partición y `buscar_consultas("fragmento")` filtra por el texto de la consulta.

In [ ]:
# Verdad de referencia y mapa consulta → texto, del caché compartido que usa el pipeline
dataset_cache_dir = cache.dataset_cache_base(
    CACHE_DIR, data.COUNTRY, data.DATASET_VERSION, data.MAX_WORD_COUNT
)
qrels, consulta_a_texto = data.get_pruned_qrels_and_queries(
    data.COUNTRY, data.DATASET_VERSION, kept_doc_ids=None,
    dataset_cache_dir=dataset_cache_dir, num_workers=1,
)
relevantes_por_consulta = qrels.to_dict()

print(f"Consultas de prueba: {len(consulta_a_texto):,}")
print(f"Verdad de referencia: {dataset_cache_dir / 'pruned_qrels.json'}")


def resolver_consulta(numero) -> str:
    """Devuelve el id de consulta (cadena) que corresponde al número dado.

    Se acepta como entero o como cadena; las claves de los runs y de los qrels son cadenas.
    """
    numero = str(numero)
    if numero not in consulta_a_texto:
        raise KeyError(
            f"La consulta {numero!r} no está en la partición de prueba. "
            "Usa listar_consultas() o buscar_consultas('...') para encontrar un número válido."
        )
    return numero


def listar_consultas(inicio: int = 0, cuantas: int = 10) -> None:
    """Imprime una rebanada de la partición, con el número y el texto de cada consulta."""
    for numero in list(consulta_a_texto)[inicio:inicio + cuantas]:
        print(f"{numero}  {consulta_a_texto[numero]}")


def buscar_consultas(fragmento: str, limite: int = 20) -> None:
    """Imprime las consultas cuyo texto contiene `fragmento` (sin distinguir mayúsculas)."""
    fragmento = fragmento.lower()
    encontradas = 0
    for numero, texto in consulta_a_texto.items():
        if fragmento in texto.lower():
            print(f"{numero}  {texto}")
            encontradas += 1
            if encontradas == limite:
                break
    print(f"\n{encontradas} consulta(s) mostradas (límite {limite}).")


# Ejemplos de uso para encontrar un número de consulta
listar_consultas(0, 5)
buscar_consultas("moneda circula en aruba")

In [ ]:
# Consulta configurada en la §1
CONSULTA = resolver_consulta(NUMERO_CONSULTA)
print(f"Consulta {CONSULTA}: {consulta_a_texto[CONSULTA]}")
for docid in relevantes_por_consulta[CONSULTA]:
    print(f"  relevante: {docid}")

## 4. Top n por modelo

Se lee el run de cada modelo (ranx lo guarda como `{consulta: {docid: score}}`), se ordenan los documentos de esa consulta por score descendente —con orden estable, para respetar el orden del run cuando hay empates— y se toman los `n` primeros. Cada run se libera antes de cargar el siguiente, de modo que no están los seis rankings en memoria a la vez.

Leer un run cuesta unos segundos (≈150 MB comprimidos, ≈2.5 GB en memoria) y se repite en cada llamada: los seis modelos tardan ≈30 s, así que conviene tener claro qué consultas se quieren inspeccionar antes de ejecutar la celda varias veces.

In [ ]:
def top_n_por_modelo(numero, n: int = N) -> dict[str, pd.DataFrame]:
    """Devuelve, por modelo, el top n de documentos recuperados para una consulta de prueba.

    Returns:
        {alias: DataFrame} con columnas: posicion, docid, puntaje, relevante.
    """
    numero = resolver_consulta(numero)
    relevantes = relevantes_por_consulta[numero]
    resultados: dict[str, pd.DataFrame] = {}

    for modelo in MODELOS:
        ruta = ruta_run(modelo)
        t0 = time.time()
        run = load_lz4(str(ruta))       # mismo dict que escribió ranx.Run.save
        if numero not in run:
            raise KeyError(f"El run de {modelo.alias} no contiene la consulta {numero}: {ruta}")

        top = sorted(run[numero].items(), key=lambda par: par[1], reverse=True)[:n]
        del run
        gc.collect()

        resultados[modelo.alias] = pd.DataFrame([
            {
                "posicion": posicion,
                "docid": docid,
                "puntaje": puntaje,
                "relevante": "sí" if docid in relevantes else "",
            }
            for posicion, (docid, puntaje) in enumerate(top, start=1)
        ])

        aviso = "" if len(top) == n else f"  (el run solo devolvió {len(top)})"
        print(f"{modelo.alias:>13}: {len(top):2d} documentos en {time.time() - t0:.1f} s{aviso}")

    return resultados

In [ ]:
# Top n de la consulta configurada. Para otra consulta, cambia el número en esta llamada:
#     resultados = top_n_por_modelo(8199550, N)
resultados = top_n_por_modelo(NUMERO_CONSULTA, N)

## 5. Textos de los documentos

Los `docid` que aparecen en las tablas (los recuperados y los relevantes) se buscan en el corpus de párrafos de la Wikipedia en español (`eswiki_20240401`), de donde se saca el título y el texto. El corpus está en la caché de HuggingFace y se lee mapeado en disco: solo se materializan los párrafos pedidos, no los 14 millones del corpus.

Si `INCLUIR_TEXTO = False`, esta sección no carga nada y las tablas de la §6 salen sin título ni extracto.

In [ ]:
# Título y texto de los párrafos que aparecen en las tablas. Se importa aquí porque solo hace
# falta para leer el corpus.
import datasets
import pyarrow as pa
import pyarrow.compute as pc

docids = sorted(
    set().union(*[set(tabla["docid"]) for tabla in resultados.values()])
    | set(relevantes_por_consulta[CONSULTA])
)

textos: dict[str, tuple[str, str]] = {}
if INCLUIR_TEXTO:
    t0 = time.time()
    corpus = datasets.load_dataset(data.CORPUS_NAME, split="corpus")
    tabla_corpus = corpus.data.table
    columna_docid = pc.cast(tabla_corpus["docid"], pa.string())
    seleccion = tabla_corpus.filter(
        pc.is_in(columna_docid, value_set=pa.array(docids, type=columna_docid.type))
    )
    for docid, titulo, texto in zip(
        seleccion["docid"].to_pylist(),
        seleccion["title"].to_pylist(),
        seleccion["text"].to_pylist(),
    ):
        textos[str(docid)] = (titulo or "", texto or "")
    print(f"Párrafos pedidos: {len(docids)} | encontrados en el corpus: {len(textos)} "
          f"({time.time() - t0:.1f} s)")
else:
    print("INCLUIR_TEXTO = False: las tablas de la §6 salen sin título ni extracto.")

## 6. Resultados

El top n de cada modelo para la consulta configurada. `posicion` es el lugar que el modelo le da al documento (1 = primero), `puntaje` es el score del run y `relevante` marca los documentos que la verdad de referencia considera relevantes para esta consulta.

Después de las tablas se imprime el **texto completo de los documentos relevantes**, para poder juzgar lo que cada modelo devolvió.

In [ ]:
def _extracto(texto: str, limite: int = 160) -> str:
    """Primeras palabras del documento, para que la tabla se pueda leer."""
    texto = " ".join(texto.split())
    if len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + " […]"


def mostrar_resultados(resultados) -> None:
    """Muestra el top n de cada modelo, con título y extracto si los textos están cargados."""
    for alias, tabla in resultados.items():
        vista = tabla.copy()
        if textos:
            vista["titulo"] = [textos.get(docid, ("", ""))[0] for docid in vista["docid"]]
            vista["extracto"] = [_extracto(textos.get(docid, ("", ""))[1]) for docid in vista["docid"]]
        display(Markdown(f"**{alias}** — top {len(vista)} de «{consulta_a_texto[CONSULTA]}»"))
        display(vista)


def mostrar_relevantes() -> None:
    """Imprime el texto completo de los documentos relevantes de la consulta."""
    display(Markdown("**Documentos relevantes (verdad de referencia)**"))
    for docid in relevantes_por_consulta[CONSULTA]:
        titulo, texto = textos.get(docid, ("(sin texto)", ""))
        print(f"{docid} — {titulo}\n{texto}\n")


mostrar_resultados(resultados)
mostrar_relevantes()

## 7. Dónde aparece cada documento relevante

Para cada documento relevante de la consulta, la posición que le asigna cada modelo dentro de su top n (`—` si no aparece en él). La última columna cuenta en cuántos de los seis modelos aparece.

In [ ]:
def resumen_relevantes(resultados) -> pd.DataFrame:
    """Posición de cada documento relevante en el top n de cada modelo ('—' si no aparece)."""
    filas = []
    for docid in relevantes_por_consulta[CONSULTA]:
        fila = {"docid": docid}
        for alias, tabla in resultados.items():
            posiciones = tabla.index[tabla["docid"] == docid]
            fila[alias] = int(tabla.loc[posiciones[0], "posicion"]) if len(posiciones) else "—"
        fila["modelos"] = sum(1 for alias in resultados if fila[alias] != "—")
        filas.append(fila)
    return pd.DataFrame(filas)


display(resumen_relevantes(resultados))